<a href="https://colab.research.google.com/github/auliatauhid/Data-Science-2026/blob/main/Pertemuan6_AuliaTauhid_250401020136.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aulia Tauhid Akbar - 250401020136 - Data Science IF405**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# ============================================================
# LANGKAH 1: LOAD & EDA SINGKAT
# ============================================================

df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = ['pclass','sex','age','sibsp','parch','fare','embarked','survived']
df = df[cols].copy()

print('Shape:', df.shape)
print('\nMissing values:')
print(df.isnull().sum())

print('\nDistribusi target:')
print(df['survived'].value_counts(normalize=True).round(3))
# survived=0: ~61.6%, survived=1: ~38.4% — kelas tidak seimbang!

Shape: (891, 8)

Missing values:
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

Distribusi target:
survived
0    0.616
1    0.384
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# LANGKAH 2: HANDLING MISSING VALUES
# ============================================================

# Age: isi dengan median (robust terhadap outlier)
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi dengan modus (nilai paling sering)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum())  # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [ ]:
# ============================================================
# LANGKAH 3: ENCODING KATEGORIKAL
# ============================================================

# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(df,
                    columns=['sex', 'embarked'],
                    drop_first=True,   # hindari dummy variable trap
                    dtype=int)

print('Kolom setelah encoding:')
print(df.columns.tolist())
# ['pclass','age','sibsp','parch','fare','survived',
#  'sex_male','embarked_Q','embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [ ]:
# ============================================================
# LANGKAH 4: TRAIN-TEST SPLIT
# ============================================================

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # proporsi kelas terjaga
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')
print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))
print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


In [ ]:
# ============================================================
# LANGKAH 5: FEATURE SCALING
# ============================================================

# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) TIDAK perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set (belajar μ dan σ dari sini)
X_train = X_train.copy()
X_test  = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform saja pada test set (gunakan μ dan σ dari training!)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))
print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))

print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)


**Kesimpulan**

Program ini membangun pipeline preprocessing ML yang lengkap dan benar — dari raw data hingga data siap dilatih model — mencakup: EDA singkat, imputasi missing values, one-hot encoding dengan drop_first, stratified train-test split, dan feature scaling dengan StandardScaler yang diterapkan secara benar (fit hanya di train, transform di test).

Temuan Utama

Data awal memiliki dua masalah utama:

age: 177 missing values (hampir 20% dari 891 baris) — diisi median (28.0)
embarked: 2 missing values — diisi modus ('S')

Target tidak seimbang (class imbalance):

61.6% tidak selamat vs 38.4% selamat — penggunaan stratify=y pada split sudah tepat untuk menjaga proporsi ini di train dan test.

Hasil split bersih dan proporsional:
TrainTestJumlah baris712179Proporsi survived38.3%38.5%
Scaling berhasil menormalisasi skala fitur — fare yang awalnya rentangnya sangat lebar (std=48) sekarang sejajar dengan fitur lain.

Keterbatasan / Pertanyaan yang Muncul

Imputasi age dengan median global (28.0) terlalu kasar — usia optimal diisi berdasarkan kelompok (misalnya median per pclass atau per title dari kolom nama), karena distribusi usia antar kelas sangat berbeda.
Kolom pclass di-scale padahal sebenarnya ordinal — pclass (1, 2, 3) lebih bermakna sebagai urutan, bukan numerik kontinu. Scaling-nya tidak salah, tapi perlu disadari maknanya berubah.
Tidak ada feature engineering — fitur seperti family_size = sibsp + parch + 1 atau ekstraksi gelar dari nama (Mr., Mrs., Miss.) terbukti meningkatkan performa model Titanic secara signifikan.
Class imbalance belum ditangani secara eksplisit — stratify menjaga proporsi split, tapi saat training model nanti perlu pertimbangan teknik seperti class_weight='balanced', SMOTE, atau threshold tuning.
Langkah natural berikutnya: melatih model klasifikasi (Logistic Regression, Random Forest, atau XGBoost) dan mengevaluasinya dengan precision, recall, F1-score, bukan hanya accuracy — karena data tidak seimbang.